# Prepare and merge all data for Autoencoder

In [1]:
import pandas as pd
import re
from datetime import datetime

In [3]:
# Variables and Paths
ALL_DATA_CSV = "output/merged_data.csv"
#LATENT_FILE = "output/Experiments/BetaScanVAE/latent_representations/train_mu_beta_9.00e-05.csv"
# LATENT_FILE = "output/Experiments/BetaScanVAE/latent_representations/train_mu_beta_3.00e-06.csv"
# LATENT_FILE = "output/Experiments/BetaScanVAE/latent_representations/train_mu_beta_3.00e-05.csv"
LATENT_FILE = "output/Experiments/BetaScanVAE/latent_representations/val_mu_beta_9.00e-05.csv"
DICOM_FILE = "data/csvData/dicom_metadata.csv"
OUTPUT_FILE = "output/final_validation_combined_vae_data_beta_9.00e-05.csv"
#OUTPUT_FILE = "output/final_validation_combined_vae_data.csv"

In [5]:
# Load merged clinical data
df_merged = pd.read_csv(ALL_DATA_CSV, low_memory=False)
print(f"Merged clinical data: {df_merged.shape}")

# Load latent vectors
df_latent = pd.read_csv(LATENT_FILE)
print(f"Latent vectors: {df_latent.shape}")

# Load DICOM metadata for scanner info
df_dicom = pd.read_csv(DICOM_FILE)
print(f"DICOM metadata: {df_dicom.shape}")

Merged clinical data: (41816, 42)
Latent vectors: (596, 259)
DICOM metadata: (2986, 6)


In [6]:
df_latent.rename(columns={'file_path': 'FilePath'}, inplace=True)
df_latent_clean = df_latent.dropna(subset=['FilePath']).copy()
df_latent_clean.shape

(596, 259)

In [7]:
df_latent.head(2)

,FilePath,PATNO,label,latent_0,latent_1,latent_2,latent_3,latent_4,latent_5,latent_6,...,latent_246,latent_247,latent_248,latent_249,latent_250,latent_251,latent_252,latent_253,latent_254,latent_255
0,data/Images/PPMI_Images_PD/42171/Reconstructed...,42171,PD,0.005189,-0.024983,0.015525,-0.004893,-0.006701,-0.037571,0.058676,...,0.020005,0.014320,-0.119046,-0.031732,0.467966,-0.109838,0.069473,-0.002921,0.020634,0.008552
1,data/Images/PPMI_Images_PD/144254/Reconstructe...,144254,PD,-0.031478,0.049452,0.079944,0.025320,0.057972,0.082475,0.182108,...,0.030731,0.029798,-0.243916,-0.003038,0.135128,0.015894,-0.024634,-0.092181,-0.011098,0.020219


In [8]:
df_dicom.rename(columns={'file_path': 'FilePath'}, inplace=True)
df_dicom_latest = df_dicom.dropna(subset=['FilePath']).copy()
df_dicom_latest.shape

(2986, 6)

In [9]:
df_dicom_latest.head(2)

,FilePath,group,PatientSex,StudyDescription,Manufacturer,ManufacturerModelName
0,Images\PPMI_Images_PD\100001\Reconstructed_DaT...,PD,M,1-DAT,SIEMENS NM,Encore2
1,Images\PPMI_Images_PD\100001\Reconstructed_DaT...,PD,M,V02-DAT,SIEMENS NM,Encore2


In [10]:
print("--- Latent DataFrame Path Example ---")
print(df_latent_clean['FilePath'].iloc[0])

print("\n--- DICOM DataFrame Path Example ---")
print(df_dicom_latest['FilePath'].iloc[0])

--- Latent DataFrame Path Example ---
data/Images/PPMI_Images_PD/42171/Reconstructed_DaTSCAN/2015-12-17_14_07_50.0/I764441/PPMI_42171_NM_Reconstructed_DaTSCAN_Br_20160808175753049_1_S482958_I764441.dcm

--- DICOM DataFrame Path Example ---
Images\PPMI_Images_PD\100001\Reconstructed_DaTSCAN\2020-09-09_17_07_33.0\I1452480\PPMI_100001_NM_Reconstructed_DaTSCAN_Br_20210608102518754_1_S1028880_I1452480.dcm


In [11]:
# Function to normalize paths
def normalize_path(path_str):
    if pd.isna(path_str): return path_str
    
    # 1. Force forward slashes
    clean_p = path_str.replace('\\', '/')
    
    # 2. Remove 'data/' prefix if it exists to ensure matching
    if clean_p.startswith('data/'):
        clean_p = clean_p.replace('data/', '')
        
    # 3. Strip any leading/trailing whitespace
    return clean_p.strip()

# Apply to BOTH dataframes
df_latent_clean['Merge_Key'] = df_latent_clean['FilePath'].apply(normalize_path)
df_dicom_latest['Merge_Key'] = df_dicom_latest['FilePath'].apply(normalize_path)

# Check if they look the same now
print("New Key Latent:", df_latent_clean['Merge_Key'].iloc[0])
print("New Key DICOM: ", df_dicom_latest['Merge_Key'].iloc[0])

New Key Latent: Images/PPMI_Images_PD/42171/Reconstructed_DaTSCAN/2015-12-17_14_07_50.0/I764441/PPMI_42171_NM_Reconstructed_DaTSCAN_Br_20160808175753049_1_S482958_I764441.dcm
New Key DICOM:  Images/PPMI_Images_PD/100001/Reconstructed_DaTSCAN/2020-09-09_17_07_33.0/I1452480/PPMI_100001_NM_Reconstructed_DaTSCAN_Br_20210608102518754_1_S1028880_I1452480.dcm


In [12]:
df_latent_with_scanner = pd.merge(
    df_latent_clean,
    df_dicom_latest[['Merge_Key', 'Manufacturer', 'ManufacturerModelName']],
    on='Merge_Key',
    how='left'  # Keep all latent vectors, add scanner info
)
df_latent_with_scanner.shape

(596, 262)

In [13]:
df_latent_with_scanner['FilePath'] = df_latent_with_scanner['Merge_Key']

In [14]:
df_latent_with_scanner.sample(5)

,FilePath,PATNO,label,latent_0,latent_1,latent_2,latent_3,latent_4,latent_5,latent_6,...,latent_249,latent_250,latent_251,latent_252,latent_253,latent_254,latent_255,Merge_Key,Manufacturer,ManufacturerModelName
177,Images/PPMI_Images_PD/3023/Reconstructed_DaTSC...,3023,PD,0.086234,0.008302,0.002518,-0.006031,0.008631,0.103694,0.017771,...,0.066966,-0.591918,0.100857,0.002333,0.010972,0.012906,0.046569,Images/PPMI_Images_PD/3023/Reconstructed_DaTSC...,Philips Healthcare,BrightView
522,Images/PPMI_Images_PD/216837/Reconstructed_DaT...,216837,PD,-0.030100,-0.030691,0.009994,-0.037065,-0.001245,0.057967,-0.137412,...,0.012247,0.289051,0.028170,-0.047243,-0.065799,0.040036,0.031126,Images/PPMI_Images_PD/216837/Reconstructed_DaT...,SIEMENS NM,Encore2
523,Images/PPMI_Images_PD/141710/Reconstructed_DaT...,141710,PD,0.055102,0.012511,0.033364,-0.007207,0.003390,0.042111,-0.013617,...,-0.016662,1.666755,0.037461,-0.014764,-0.033137,-0.025375,0.026905,Images/PPMI_Images_PD/141710/Reconstructed_DaT...,SIEMENS NM,Encore2
250,Images/PPMI_Images_PD/100898/Reconstructed_DaT...,100898,PD,0.044273,-0.099569,-0.074347,0.012411,-0.036436,0.146942,0.095003,...,0.011657,-0.422246,-0.092687,0.131386,-0.008834,0.035980,0.094878,Images/PPMI_Images_PD/100898/Reconstructed_DaT...,SIEMENS NM,Encore2
454,Images/PPMI_Images_PD/40714/Reconstructed_DaTS...,40714,PD,-0.056429,-0.030396,-0.003836,-0.038180,0.012454,0.032428,-0.057450,...,0.044730,-0.995270,-0.025310,0.017210,-0.029781,-0.045920,0.020002,Images/PPMI_Images_PD/40714/Reconstructed_DaTS...,GE MEDICAL SYSTEMS,INFINIA


In [15]:
# 1. Robust Date Extraction (Finds YYYY-MM-DD anywhere in path)
def get_date_from_path(path_str):
    if pd.isna(path_str):
        return None
    
    # Regex to find pattern: 4 digits - 2 digits - 2 digits
    match = re.search(r'(\d{4}-\d{2}-\d{2})', str(path_str))
    if match:
        date_raw = match.group(1) # Extracts '2021-04-06'
        try:
            # Added datetime import requirement and better error handling
            return datetime.strptime(date_raw, '%Y-%m-%d').strftime('%m/%Y')
        except Exception:
            return None
    return None

In [16]:
# 2. Apply the fix
df_latent_with_scanner['DATSCAN_DATE'] = df_latent_with_scanner['FilePath'].apply(get_date_from_path)

# Verify we actually have dates now (Safe check)
dates_found = df_latent_with_scanner['DATSCAN_DATE'].dropna()
if not dates_found.empty:
    print("Latent Date Sample:", dates_found.iloc[0])
else:
    print("Warning: No dates could be extracted from FilePath. Check your regex or path format.")

df_latent_with_scanner.sample(2)

Latent Date Sample: 12/2015


,FilePath,PATNO,label,latent_0,latent_1,latent_2,latent_3,latent_4,latent_5,latent_6,...,latent_250,latent_251,latent_252,latent_253,latent_254,latent_255,Merge_Key,Manufacturer,ManufacturerModelName,DATSCAN_DATE
502,Images/PPMI_Images_PD/74067/Reconstructed_DaTS...,74067,PD,-0.037089,0.017082,-0.025881,-0.041939,0.027588,0.064690,0.060515,...,1.060272,0.079743,0.049213,-0.033713,0.020286,0.053854,Images/PPMI_Images_PD/74067/Reconstructed_DaTS...,"Picker International, NM Division",P3000XP,05/2022
152,Images/PPMI_Images_PD/171001/Reconstructed_DaT...,171001,PD,-0.016419,0.052418,0.063358,0.037124,0.011381,0.062361,0.163654,...,0.641410,0.113991,-0.005302,-0.085957,-0.063076,0.064730,Images/PPMI_Images_PD/171001/Reconstructed_DaT...,Philips Healthcare,BrightView,09/2022


In [17]:

# 3. Clean Clinical Data (df_merged)
df_merged['PATNO'] = pd.to_numeric(df_merged['PATNO'], errors='coerce').fillna(0).astype(int)
df_latent_with_scanner['PATNO'] = pd.to_numeric(df_latent_with_scanner['PATNO'], errors='coerce').fillna(0).astype(int)

In [18]:
df_latent_with_scanner.sample(2)

,FilePath,PATNO,label,latent_0,latent_1,latent_2,latent_3,latent_4,latent_5,latent_6,...,latent_250,latent_251,latent_252,latent_253,latent_254,latent_255,Merge_Key,Manufacturer,ManufacturerModelName,DATSCAN_DATE
415,Images/PPMI_Images_PD/101179/Reconstructed_DaT...,101179,PD,0.063372,0.004369,0.015754,0.034705,0.016450,0.016546,0.019045,...,0.918307,-0.070777,0.008380,-0.005258,-0.008740,0.035868,Images/PPMI_Images_PD/101179/Reconstructed_DaT...,SIEMENS NM,ENCORE2,03/2021
137,Images/PPMI_Images_PD/3102/Reconstructed_DaTSC...,3102,PD,-0.001245,-0.004846,-0.021767,-0.018560,0.026269,0.025684,-0.041406,...,-0.177642,0.048434,-0.022752,-0.012585,-0.004635,0.043555,Images/PPMI_Images_PD/3102/Reconstructed_DaTSC...,SIEMENS NM,IP2,10/2010


In [19]:
df_latent_with_scanner.shape

(596, 263)

In [20]:
# Convert clinical dates to strings, handle NaNs
df_merged['DATSCAN_DATE'] = pd.to_datetime(
    df_merged['DATSCAN_DATE'], 
    format='mixed', 
    errors='coerce'
).dt.strftime('%m/%Y')

df_merged.sample(2)

,PATNO,EVENT_ID,AGE_AT_VISIT,BIRTHDT,SEX,INFODT,REC_ID,PAG_NAME,AFICBERB,ASHKJEW,...,DATSCAN_DATE,DATSCAN_CAUDATE_R,DATSCAN_CAUDATE_L,DATSCAN_PUTAMEN_R,DATSCAN_PUTAMEN_L,DATSCAN_PUTAMEN_R_ANT,DATSCAN_PUTAMEN_L_ANT,DATSCAN_ANALYZED,DATSCAN_NOT_ANALYZED_REASON,DATSCAN_OTHER_SPECIFY
16145,42438,V10,65.6,08/1955,0.0,09/2016,IA88709,SCREEN,0.0,1.0,...,06/2021,1.62,1.98,0.35,0.46,1.0,1.29,Yes,NaN,NaN
30168,170343,V02,77.9,05/1945,1.0,11/2022,IA201319,SCREEN,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
df_merged.shape

(41816, 42)

In [22]:
# 4. Perform the Merge
df_combined = pd.merge(
    df_latent_with_scanner, 
    df_merged,
    on=['PATNO', 'DATSCAN_DATE'],
    how='inner'
)

print(f"Merge Shape: {df_combined.shape}")

Merge Shape: (591, 303)


In [23]:
df_combined['DATSCAN_DATE'].sample(5)

8      09/2022
137    04/2013
274    03/2023
150    04/2013
372    02/2018
Name: DATSCAN_DATE, dtype: object

In [24]:
df_combined.sample(5)

,FilePath,PATNO,label,latent_0,latent_1,latent_2,latent_3,latent_4,latent_5,latent_6,...,DATSCAN_LIGAND,DATSCAN_CAUDATE_R,DATSCAN_CAUDATE_L,DATSCAN_PUTAMEN_R,DATSCAN_PUTAMEN_L,DATSCAN_PUTAMEN_R_ANT,DATSCAN_PUTAMEN_L_ANT,DATSCAN_ANALYZED,DATSCAN_NOT_ANALYZED_REASON,DATSCAN_OTHER_SPECIFY
140,Images/PPMI_Images_PD/101047/Reconstructed_DaT...,101047,PD,-0.001228,-0.015074,0.010949,0.001917,-0.038799,0.148798,0.106309,...,123I-DaTscan,1.78,1.09,0.82,0.54,1.18,0.92,Yes,NaN,NaN
379,Images/PPMI_Images_PD/51632/Reconstructed_DaTS...,51632,PD,0.068658,-0.004291,-0.001060,0.009574,-0.017655,0.020182,0.097073,...,NaN,1.55,1.67,0.62,0.75,1.01,1.25,Yes,NaN,NaN
269,Images/PPMI_Images_PD/3179/Reconstructed_DaTSC...,3179,PD,0.029878,-0.029021,0.014675,0.004064,0.024346,0.092718,0.044900,...,NaN,1.38,1.98,0.50,0.72,0.86,1.36,Yes,NaN,NaN
482,Images/PPMI_Images_PD/109619/Reconstructed_DaT...,109619,PD,0.009584,-0.015788,0.030349,0.005534,-0.001020,0.056191,-0.005465,...,123I-DaTscan,2.08,1.97,1.06,0.76,1.65,1.24,Yes,NaN,NaN
405,Images/PPMI_Images_PD/141031/Reconstructed_DaT...,141031,PD,0.040703,-0.019549,0.032591,-0.055074,-0.018011,0.165077,0.249362,...,123I-DaTscan,2.05,1.62,0.67,0.60,1.08,1.03,Yes,NaN,NaN


In [25]:
"Manufacturer" in df_combined.columns

True

In [26]:
# Save the final merged dataset for the Autoencoder

df_combined.to_csv(OUTPUT_FILE, index=False)

print(f"Successfully saved merged data to: {OUTPUT_FILE}")
print(f"Final file contains {df_combined.shape[0]} rows and {df_combined.shape[1]} columns.")

Successfully saved merged data to: output/final_validation_combined_vae_data_beta_9.00e-05.csv
Final file contains 591 rows and 303 columns.
